# Bayesian Optimisation Analysis

Analyses the GP-based Bayesian optimisation run over the three-dimensional parameter space of the Custom Trader.

| Parameter | Symbol | Search range |
|---|---|---|
| Learning rate | alpha | [0.001, 0.30] |
| Position limit | L | [20, 800] |
| Volatility threshold | sigma_v | [3.0, 15.0] |

Surrogate: Gaussian Process with Matern-5/2 kernel.
Acquisition: Expected Improvement (EI).
Budget: 20 random initialisations + 80 GP-guided iterations (100 total).

Reads from `data/bayesian_search_log.csv` and `data/inventory_timeseries.csv`.
Run from the `bse/` root directory.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.lines import Line2D
from mpl_toolkits.mplot3d import Axes3D
from scipy import stats
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern, WhiteKernel, ConstantKernel
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'text.usetex':       False,
    'font.family':       'serif',
    'font.serif':        ['DejaVu Serif'],
    'axes.labelsize':    14,
    'font.size':         11,
    'legend.fontsize':   9,
    'xtick.labelsize':   12,
    'ytick.labelsize':   12,
    'axes.titlesize':    12,
    'figure.dpi':        150,
    'savefig.dpi':       300,
    'axes.spines.top':   False,
    'axes.spines.right': False,
    'axes.grid':         True,
    'grid.alpha':        0.3,
    'grid.linestyle':    '--',
    'grid.linewidth':    0.5,
})

INIT_CASH = 10000

CKPT_PATH = 'data/bayesian_search_log.csv'
SAVE_DIR  = 'figures'
os.makedirs(SAVE_DIR, exist_ok=True)

df = pd.read_csv(CKPT_PATH)
df.rename(columns={'beta': 'alpha'}, inplace=True)
df['profit_norm'] = df['mean_profit'] / INIT_CASH

N_INIT   = 20
df['phase'] = df['iteration'].apply(lambda i: 'Initialisation' if i <= N_INIT else 'GP-guided')

PARAMS       = ['alpha', 'position_limit', 'volatility_threshold']
PARAM_LABELS = {
    'alpha':                r'$\alpha$',
    'position_limit':       r'$L$',
    'volatility_threshold': r'$\sigma_v$',
}
BOUNDS = {
    'alpha':                (0.001, 0.30),
    'position_limit':       (20,    800),
    'volatility_threshold': (3.0,   15.0),
}
PHASE_COLORS = {'Initialisation': '#95a5a6', 'GP-guided': '#2980b9'}

print(f'Loaded {len(df)} iterations ({(df.phase=="Initialisation").sum()} init, {(df.phase=="GP-guided").sum()} GP-guided)')
best = df.loc[df['mean_profit'].idxmax()]
print(f'Best: alpha={best["alpha"]:.4f}, L={int(best["position_limit"])}, sigma_v={best["volatility_threshold"]:.3f}, profit_norm={best["profit_norm"]:.4f}')

## Figure 2: GP Parameter Space

In [ ]:
fig = plt.figure(figsize=(10, 7))
ax3d = fig.add_subplot(111, projection='3d')

sc = ax3d.scatter(
    df['alpha'], df['volatility_threshold'], df['position_limit'],
    c=df['iteration'], cmap='viridis', s=45, alpha=0.82
)
ax3d.scatter(
    best['alpha'], best['volatility_threshold'], best['position_limit'],
    color='red', s=300, marker='*', zorder=5, label='Optimal'
)
ax3d.set_xlabel(r'$\alpha$', size=18, labelpad=8)
ax3d.set_ylabel(r'$\sigma_v$', size=18, labelpad=8)
ax3d.set_zlabel(r'$L$', size=18, labelpad=8)
ax3d.set_title('Parameter Space Explored by the GP', size=14, fontweight='bold', pad=10)
plt.colorbar(sc, ax=ax3d, label='Iteration', shrink=0.6, pad=0.1)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'fig2_gp_parameter_space.png'), dpi=300, bbox_inches='tight')
plt.show()

## Marginal Effects of Each Parameter

Each panel shows normalised profit as a function of one parameter, with all other parameters pooled. The rolling mean highlights the trend. Dashed lines mark the search boundaries.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))

for ax, param in zip(axes, PARAMS):
    x = df[param].values
    y = df['profit_norm'].values

    for phase, grp in df.groupby('phase'):
        ax.scatter(grp[param], grp['profit_norm'],
                   color=PHASE_COLORS[phase], s=22, alpha=0.65,
                   label=phase, edgecolors='none')

    sorted_idx = np.argsort(x)
    xs, ys = x[sorted_idx], y[sorted_idx]
    if len(xs) >= 10:
        roll = pd.Series(ys).rolling(window=10, center=True, min_periods=5).mean()
        ax.plot(xs, roll.values, color='black', linewidth=2, zorder=5)

    for bnd in BOUNDS[param]:
        ax.axvline(bnd, color='grey', linewidth=0.8, linestyle='--', alpha=0.6)

    r, p = stats.pearsonr(x, y)
    sig = '*' if p < 0.05 else ''
    ax.set_xlabel(PARAM_LABELS[param])
    ax.set_ylabel(r'$P\,/\,C_0$')
    ax.set_title(f'{PARAM_LABELS[param]}   r = {r:.3f}{sig}')

axes[0].legend(frameon=False, fontsize=9)
plt.tight_layout()
plt.show()

## GP Surrogate Surface

A Gaussian Process is re-fitted to the search log data and used to visualise the predicted profit landscape over alpha x L, with sigma_v fixed at the near-optimal value of 15.0.

In [ ]:
X_raw = df[['alpha', 'position_limit', 'volatility_threshold']].values
y_raw = df['profit_norm'].values

scaler_x = StandardScaler()
X_scaled = scaler_x.fit_transform(X_raw)
y_mean, y_std = y_raw.mean(), y_raw.std()
y_scaled = (y_raw - y_mean) / y_std

kernel = ConstantKernel(1.0) * Matern(nu=2.5, length_scale=np.ones(3)) + WhiteKernel(noise_level=0.1)
gp = GaussianProcessRegressor(kernel=kernel, n_restarts_optimizer=5,
                               normalize_y=False, random_state=42)
gp.fit(X_scaled, y_scaled)

N_GRID     = 60
alpha_grid = np.linspace(BOUNDS['alpha'][0], BOUNDS['alpha'][1], N_GRID)
pos_grid   = np.linspace(BOUNDS['position_limit'][0], BOUNDS['position_limit'][1], N_GRID)
AA, PP     = np.meshgrid(alpha_grid, pos_grid)
theta_fix  = 15.0

grid_pts  = np.column_stack([AA.ravel(), PP.ravel(),
                              np.full(N_GRID * N_GRID, theta_fix)])
grid_sc   = scaler_x.transform(grid_pts)
mu_sc, sd_sc = gp.predict(grid_sc, return_std=True)
mu = mu_sc * y_std + y_mean
sd = sd_sc * y_std

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

im0 = axes[0].contourf(AA, PP, mu.reshape(N_GRID, N_GRID), levels=20, cmap='RdYlGn')
axes[0].scatter(df['alpha'], df['position_limit'], c='white', s=12, alpha=0.6, edgecolors='k', linewidth=0.4)
axes[0].scatter(best['alpha'], best['position_limit'], color='red', s=200, marker='*', zorder=5)
plt.colorbar(im0, ax=axes[0], label=r'$P\,/\,C_0$ (predicted)')
axes[0].set_xlabel(r'$\alpha$')
axes[0].set_ylabel(r'$L$')
axes[0].set_title(r'GP mean ($\sigma_v = 15.0$)')

im1 = axes[1].contourf(AA, PP, sd.reshape(N_GRID, N_GRID), levels=20, cmap='Blues')
plt.colorbar(im1, ax=axes[1], label='Predictive std')
axes[1].set_xlabel(r'$\alpha$')
axes[1].set_ylabel(r'$L$')
axes[1].set_title('GP uncertainty')

plt.tight_layout()
plt.show()

## Top 10 Parameter Combinations

In [ ]:
top10 = (df.sort_values('mean_profit', ascending=False)
           .head(10)
           [['iteration', 'alpha', 'position_limit', 'volatility_threshold', 'profit_norm', 'phase']]
           .rename(columns={'profit_norm': 'P/C_0'})
           .reset_index(drop=True))
top10.index = top10.index + 1
top10['alpha'] = top10['alpha'].round(4)
top10['P/C_0'] = top10['P/C_0'].round(4)
top10

## Figure 3: Inventory Trajectories (CT, ZIP, PRSH)

In [ ]:
CONDITIONS  = ['stable', 'shock_up', 'shock_down', 'multi_regime']
COND_LABELS = ['Stable', 'Shock Up', 'Shock Down', 'Multi-Regime']
COLORS = {
    'ZIP': '#c0392b', 'PRSH': '#1a3a5c', 'CUSTOM': '#2980b9',
    'ZIC': '#27ae60', 'SHVR': '#e67e22', 'SNPR': '#8e44ad', 'GVWY': '#95a5a6',
}
FOCUS = ['CUSTOM', 'ZIP', 'PRSH']

inv_df = pd.read_csv('data/inventory_timeseries.csv')
inv_df = inv_df[inv_df['ttype'].isin(FOCUS)].copy()

fig, axes = plt.subplots(2, 2, figsize=(13, 9), sharex=True)
axes = axes.flatten()

for ax, cond, label in zip(axes, CONDITIONS, COND_LABELS):
    sub = inv_df[inv_df['condition'] == cond]
    agg = sub.groupby(['ttype', 'timestep'])['mean_inv'].mean().reset_index()
    for t in FOCUS:
        ts = agg[agg.ttype == t].sort_values('timestep')
        ax.plot(ts['timestep'] * 10, ts['mean_inv'],
                color=COLORS[t], linewidth=2.0, label=t)
    ax.axhline(0, color='black', linewidth=0.8, alpha=0.4)
    ax.set_title(label)
    ax.set_ylabel('Mean inventory (units)')
    ax.set_xlabel('Time step')
    if cond in ('shock_up', 'shock_down'):
        ax.axvline(1200, color='grey', linewidth=0.8, linestyle='--', alpha=0.7)
    axes[0].legend(frameon=False)

fig.suptitle('Mean Inventory Trajectories', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'fig3_inventory_trajectories.png'), dpi=300, bbox_inches='tight')
plt.show()